# EDA and results: NL housing forecast

This notebook explores the synthetic dataset generated by `housing generate-sample`
and, if you have a live BigQuery project set up (see the README), pulls in the
`ML.EVALUATE` and `ML.FORECAST` results for both model families.

Run the setup cell first; everything after that works whether or not GCP
credentials are configured, since it falls back to the local sample CSV.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from housing.data_gen import GenConfig, generate_rows

sample_path = Path("../data/sample/regional_price_index.csv")
if sample_path.exists():
    df = pd.read_csv(sample_path, parse_dates=["month"])
else:
    rows = generate_rows(GenConfig(regions=12, months=96))
    df = pd.DataFrame(rows)
    df["month"] = pd.to_datetime(df["month"])

df.head()

## Price trend by region

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for region, g in df.groupby("region"):
    ax.plot(g["month"], g["avg_sale_price_eur"], label=region)
ax.set_title("Average sale price by region")
ax.set_ylabel("EUR")
ax.legend(fontsize=8, ncol=2)
plt.show()

## Mortgage rate vs. national average price

Sanity check that the synthetic rate shock (months ~60-78) coincides with a
slowdown in price growth, the way it does in the real 2022-2023 Dutch market.

In [ ]:
national = df.groupby("month").agg(
    avg_sale_price_eur=("avg_sale_price_eur", "mean"),
    mortgage_rate_pct=("mortgage_rate_pct", "mean"),
).reset_index()

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(national["month"], national["avg_sale_price_eur"], color="tab:blue", label="Avg price")
ax1.set_ylabel("Avg sale price (EUR)", color="tab:blue")

ax2 = ax1.twinx()
ax2.plot(national["month"], national["mortgage_rate_pct"], color="tab:red", label="Mortgage rate")
ax2.set_ylabel("Mortgage rate (%)", color="tab:red")

fig.suptitle("National average price vs. mortgage rate")
plt.show()

## Pulling in BigQuery ML results (requires a live project)

This cell is skipped gracefully if `GOOGLE_CLOUD_PROJECT` isn't set or the
models haven't been trained yet.

In [ ]:
import os

if os.environ.get("GOOGLE_CLOUD_PROJECT"):
    from housing.config import load_settings
    from housing.sql_templates import render
    from housing.bq_client import query_to_dataframe

    settings = load_settings()
    try:
        sql = render(
            "queries/evaluate_models.sql",
            settings,
            arima_ref=settings.model_ref("nl_housing_arima"),
            boosted_tree_ref=settings.model_ref("nl_housing_boosted_tree"),
        )
        eval_df = query_to_dataframe(sql, settings)
        display(eval_df)
    except Exception as exc:
        print(f"Skipping live BigQuery pull: {exc}")
else:
    print("GOOGLE_CLOUD_PROJECT not set; skipping live BigQuery pull.")